In [1]:
!pip3 install langchain chromadb faiss-cpu tiktoken langchain_huggingface langchain-community wikipedia


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.14/bin/python3.14 -m pip install --upgrade pip


In [5]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [6]:
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import  Document
from langchain_huggingface import  ChatHuggingFace,HuggingFaceEndpoint
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [10]:
embedding_model=HuggingFaceEmbeddings()
vector_store=FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [12]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [16]:
import os
# The simplest and most reliable way: set the environment variable
import getpass
os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("<REDACTED_HF_TOKEN>")
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    max_new_tokens=512
)
model=ChatHuggingFace(llm=llm)
compressor=LLMChainExtractor.from_llm(model)


In [14]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [15]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)
compressed_results

[Document(metadata={'source': 'Doc1'}, page_content='Photosynthesis is the process by which green plants convert sunlight into energy.'),
 Document(metadata={'source': 'Doc4'}, page_content='Photosynthesis does not occur in animal cells.'),
 Document(metadata={'source': 'Doc2'}, page_content='The chlorophyll in plant cells captures sunlight during photosynthesis.')]